In [ ]:
import requests
import re
import json
import subprocess
import sys
from Bio import SeqIO


def http_function(endpoint, **http_args):
    try:
        resp = requests.get(endpoint, **http_args)
        return resp
    except requests.exceptions.RequestException as e:
        class ErrorResponse:
            def __init__(self, error):
                self.status_code = 500
                self._error = error
            def json(self):
                return {"error": str(self._error)}
            def raise_for_status(self):
                raise requests.exceptions.RequestException(self._error)
        return ErrorResponse(e)


def get_uniprot(accession):
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
    headers = {"Accept": "application/json"}
    return http_function(endpoint, headers=headers)


def uniprot_parse_response(resp):
    try:
        if resp.status_code != 200:
            try:
                error_data = resp.json()
                if "messages" in error_data:
                    return {"error": error_data["messages"][0]}
                else:
                    return {"error": f"HTTP Error {resp.status_code}"}
            except:
                return {"error": f"HTTP Error {resp.status_code}"}
        
        data = resp.json()
        accession = data.get("primaryAccession")
        
        organism = data.get("organism", {})
        organism_name = organism.get("scientificName") if organism else None
        
        genes = data.get("genes", [])
        gene_info = []
        for gene in genes:
            gene_entry = {}
            if "geneName" in gene:
                gene_entry["geneName"] = gene["geneName"]
            if "synonyms" in gene:
                gene_entry["synonyms"] = gene["synonyms"]
            if gene_entry:
                gene_info.append(gene_entry)
        
        sequence = data.get("sequence", {})
        sequence_info = {
            "value": sequence.get("value", ""),
            "length": sequence.get("length", 0),
            "molWeight": sequence.get("molWeight", 0),
            "crc64": sequence.get("crc64", ""),
            "md5": sequence.get("md5", "")
        } if sequence else None
        
        protein_type = "protein"
        if "proteinDescription" in data:
            prot_desc = data["proteinDescription"]
            if "recommendedName" in prot_desc and "fullName" in prot_desc["recommendedName"]:
                protein_type = prot_desc["recommendedName"]["fullName"].get("value", "protein")
            elif "submittedName" in prot_desc and len(prot_desc["submittedName"]) > 0:
                protein_type = prot_desc["submittedName"][0].get("fullName", {}).get("value", "protein")
        
        output = {
            accession: {
                "organism": organism_name,
                "geneInfo": gene_info if gene_info else None,
                "sequenceInfo": sequence_info,
                "type": protein_type
            }
        }
        return output
        
    except Exception as e:
        return {"error": str(e)}


def get_ensembl(id):
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    headers = {"Content-Type": "application/json"}
    params = {"expand": 1}
    return http_function(endpoint, headers=headers, params=params)


def ensembl_parse_response(resp):
    try:
        if resp.status_code != 200:
            try:
                error_data = resp.json()
                if "error" in error_data:
                    return {"error": error_data["error"]}
                else:
                    return {"error": f"HTTP Error {resp.status_code}"}
            except:
                return {"error": f"HTTP Error {resp.status_code}"}
        
        data = resp.json()
        ensembl_id = data.get("id")
        
        output = {
            ensembl_id: {
                "object_type": data.get("object_type"),
                "assembly_name": data.get("assembly_name"),
                "species": data.get("species"),
                "db_type": data.get("db_type"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "id": data.get("id"),
                "description": data.get("description"),
                "canonical_transcript": data.get("canonical_transcript"),
                "source": data.get("source")
            }
        }
        
        return output
        
    except Exception as e:
        return {"error": str(e)}


def identify_database(id):
    uniprot_patterns = [
        r'^[OPQ][0-9][A-Z0-9]{3}[0-9]$',
        r'^[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]{0,3}$',
        r'^[A-NR-Z][0-9]{5}$',
    ]
    
    ensembl_patterns = [
        r'^ENS[A-Z]*G\\d{11}$',
        r'^ENS[A-Z]*T\\d{11}$',
        r'^ENS[A-Z]*P\\d{11}$',
        r'^ENS[A-Z]*E\\d{11}$',
    ]
    
    for pattern in uniprot_patterns:
        if re.match(pattern, id):
            return "uniprot"
    
    for pattern in ensembl_patterns:
        if re.match(pattern, id):
            return "ensembl"
    
    return "unknown"


class MyFastaParser:
    
    _UNIPROT_RE = re.compile(r'sp\|([A-Z0-9]+)\|([A-Z0-9_]+)')
    _ENSEMBL_RE = re.compile(r'(ENS[A-Z]*T\d{11})(?:\.\d+)?')

    def __init__(self, file_name: str):
        self.filename = file_name

    def _get_uniprot(self, accession: str) -> dict:
        resp = get_uniprot(accession)
        parsed = uniprot_parse_response(resp)
        if "error" in parsed:
            return parsed
        return parsed.get(accession, parsed)

    def _get_ensembl(self, id: str) -> dict:
        resp = get_ensembl(id)
        parsed = ensembl_parse_response(resp)
        if "error" in parsed:
            return parsed
        return parsed.get(id, parsed)

    def _access_database(self, seq_id: str, database: str, seq_description: str, seq_sequence: str) -> dict:
        result = {
            f'file_info_{seq_id}': {
                'description': seq_description,
                'sequence': seq_sequence
            },
            f'database_info_{seq_id}': {}
        }
        
        try:
            if database == 'uniprot':
                db_info = self._get_uniprot(seq_id)
                result[f'database_info_{seq_id}'] = db_info
                result['DB_name'] = 'uniprot'
            elif database == 'ensembl':
                db_info = self._get_ensembl(seq_id)
                result[f'database_info_{seq_id}'] = db_info
                result['DB_name'] = 'ensembl'
        except Exception as e:
            result[f'database_info_{seq_id}'] = {'WARNING': f'Error: {str(e)}'}
        
        return result

    def seqkit_stats(self) -> dict:
        try:
            proc = subprocess.run(
                ['seqkit', 'stats', self.filename, '-a'],
                capture_output=True,
                text=True,
                check=False
            )
            
            if proc.returncode == 127:
                return {'error': 'seqkit not found. Please install seqkit first.'}
            
            if proc.stderr and 'no such file' in proc.stderr.lower():
                return {'error': proc.stderr.strip()}
            
            if proc.returncode != 0 or 'error' in proc.stderr.lower():
                clean_err = re.sub(r'\x1b\[[0-9;]*m', '', proc.stderr).strip()
                return {'error': clean_err if clean_err else 'SeqKit encountered an error'}
            
            lines = proc.stdout.strip().split('\n')
            if len(lines) < 2:
                return {'error': 'SeqKit returned no data'}
            
            headers = lines[0].split()
            values = lines[1].split()
            
            stat_dict = {}
            for i in range(1, len(headers)):
                if i < len(values):
                    stat_dict[headers[i]] = values[i]
            
            return {
                'fasta_seqkit_stat_info': stat_dict,
                'fasta_type': stat_dict.get('type', 'Unknown'),
                'fasta_num_seqs': int(stat_dict.get('num_seqs', 0))
            }
            
        except FileNotFoundError:
            return {'error': 'seqkit command not found. Please install seqkit first.'}
        except Exception as e:
            return {'error': str(e)}

    def biopython_parser(self, seqkit_result: dict) -> dict:
        if 'error' in seqkit_result:
            return {'error': seqkit_result['error']}
        
        fasta_type = seqkit_result.get('fasta_type', '')
        
        if fasta_type == 'Protein':
            id_pattern = self._UNIPROT_RE
            database = 'uniprot'
        else:
            id_pattern = self._ENSEMBL_RE
            database = 'ensembl'
        
        output = {'DB_name': database}
        warnings = []
        
        try:
            sequences = list(SeqIO.parse(self.filename, 'fasta'))
            
            for record in sequences:
                description = record.description
                sequence = str(record.seq)
                
                match = id_pattern.search(description)
                
                if match:
                    if database == 'uniprot':
                        seq_id = match.group(2)
                    else:
                        seq_id = match.group(1)
                    
                    seq_data = self._access_database(seq_id, database, description, sequence)
                    output.update(seq_data)
                else:
                    warnings.append(f'No ID match found for: {description[:50]}...')
                    output[f'file_info_unknown'] = {
                        'description': description,
                        'sequence': sequence
                    }
            
            if warnings:
                output['WARNING'] = warnings
            
            return output
            
        except FileNotFoundError:
            return {'error': f'File {self.filename} not found'}
        except Exception as e:
            return {'error': f'Error parsing FASTA file: {str(e)}'}

    def show_output(self, output: dict, indent: int = 0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, dict):
                        self.show_output(item, indent + 1)
                    else:
                        print('\t' * (indent + 1) + str(item))
            else:
                if key == 'sequence' and isinstance(value, str) and len(value) > 100:
                    print('\t' * (indent + 1) + value)
                else:
                    print('\t' * (indent + 1) + str(value))


print("="*80)
print("TESTING FASTA PARSER ON ALL PROVIDED FILES")
print("="*80)

test_files = [
    'test_file.fasta',
    'uniprot_download.fasta',
    'ensembl_download_1.fasta',
    'ensembl_download_2.fasta'
]

for idx, filename in enumerate(test_files, 1):
    print(f"\n{'='*80}")
    print(f"TEST {idx}: {filename}")
    print('='*80)
    
    parser = MyFastaParser(filename)
    
    print("\n1. RUNNING SEQKIT STATS...")
    stats = parser.seqkit_stats()
    
    if 'error' in stats:
        print(f"\nERROR DETECTED:")
        parser.show_output(stats)
        print("\n" + "-"*80)
        print("SKIPPING BIOPYTHON PARSER DUE TO ERROR")
        print("-"*80)
        continue
    
    print("\nSEQKIT STATS SUCCESSFUL:")
    parser.show_output(stats)
    
    print("\n2. RUNNING BIOPYTHON PARSER...")
    result = parser.biopython_parser(stats)
    
    if 'error' in result:
        print(f"\nERROR IN BIOPYTHON PARSER:")
        parser.show_output(result)
    else:
        print("\nBIOPYTHON PARSER OUTPUT:")
        parser.show_output(result)
    
    print("\n" + "-"*80)

print("\n" + "="*80)
print("ALL TESTS COMPLETED")
print("="*80)